# GRPO-Zero — analysis: before vs. after

Reads the artifacts produced by the training run (`results/metrics.json`, `results/train_log.json`, and the saved completions) and shows:
1. baseline vs. GRPO-trained pass@1 (+ delta),
2. the reward / KL training curve,
3. side-by-side sample completions the trained policy *fixed* (baseline wrong → trained right).

Run it after the training notebook has produced `results/`. Cells degrade gracefully if a file is missing yet.

In [ ]:
import json, os

# Find the repo's results/ dir whether this runs from repo root or notebooks/.
ROOT = '.' if os.path.isdir('results') else ('..' if os.path.isdir('../results') else '.')
def rpath(name): return os.path.join(ROOT, 'results', name)
def load_json(name):
    p = rpath(name)
    if not os.path.exists(p):
        print(f'(missing) {p} — run the training notebook first'); return None
    with open(p) as f: return json.load(f)

## 1. Headline: baseline vs. trained pass@1

In [ ]:
m = load_json('metrics.json') or {}
base = m.get('baseline_full') or m.get('baseline')
trained = m.get('trained')
if base:
    print(f"baseline  pass@1 = {base['accuracy']:.4f}  (model={base['model']}, n={base['n']})")
if trained:
    print(f"trained   pass@1 = {trained['accuracy']:.4f}  (n={trained['n']})")
if base and trained:
    d = trained['accuracy'] - base['accuracy']
    print(f"delta            = {d:+.4f}  ({d/base['accuracy']*100:+.1f}% relative)")

## 2. Training curve — mean reward and KL vs. step

In [ ]:
import matplotlib.pyplot as plt

hist = load_json('train_log.json') or []
if hist:
    steps = [h['step'] for h in hist]
    fig, ax1 = plt.subplots(figsize=(8, 5))
    ax1.plot(steps, [h['reward_mean'] for h in hist], 'C0', label='mean reward')
    ax1.plot(steps, [h['frac_correct'] for h in hist], 'C2--', alpha=0.5, label='frac correct')
    ax1.set_xlabel('training step'); ax1.set_ylabel('reward / accuracy', color='C0')
    ax2 = ax1.twinx()
    ax2.plot(steps, [h['kl'] for h in hist], 'C1', alpha=0.6, label='KL to ref')
    ax2.set_ylabel('KL to reference', color='C1')
    lines = ax1.get_lines() + ax2.get_lines()
    ax1.legend(lines, [l.get_label() for l in lines], loc='lower right', fontsize=9)
    plt.title('GRPO on GSM8K (RLVR): reward and KL vs. step')
    fig.tight_layout(); plt.show()

## 3. Before / after — problems the trained policy fixed
Matched by position (baseline and trained eval iterate the same test set in the same order).

In [ ]:
base_c = load_json('baseline_completions_full.json') or load_json('baseline_completions.json')
trained_c = load_json('trained_completions.json')

if base_c and trained_c:
    n = min(len(base_c), len(trained_c))
    fixed     = [(b, t) for b, t in zip(base_c[:n], trained_c[:n]) if not b['correct'] and t['correct']]
    regressed = [(b, t) for b, t in zip(base_c[:n], trained_c[:n]) if b['correct'] and not t['correct']]
    print(f'over {n} problems: fixed (wrong->right) = {len(fixed)} | regressed (right->wrong) = {len(regressed)}')
    print(f'net = {len(fixed) - len(regressed):+d}\n')
    for b, t in fixed[:3]:
        print('=' * 90)
        print('Q   :', b['question'][:280].replace('\n', ' '))
        print('GOLD:', b['gold'])
        print(f"--- BASELINE  (wrong, pred={b['pred']}) ---")
        print(b['completion'][:700].strip())
        print(f"--- TRAINED   (right, pred={t['pred']}) ---")
        print(t['completion'][:700].strip())
        print()